In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
import pandas as pd

file1 = "/home/user/Downloads/Bitext_Sample_Customer_Service_Validation_Dataset.csv"
file2 = "/home/user/Downloads/Bitext_Sample_Customer_Service_Training_Dataset.csv"
file3 = "/home/user/Downloads/Bitext_Sample_Customer_Service_Testing_Dataset.csv"

df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)
df3 = pd.read_csv(file3)

print(df1.shape)
print(df2.shape)
print(df3.shape)

(818, 4)
(6539, 4)
(818, 4)


In [5]:
df = pd.concat(
    [df1, df2, df3],
    axis=0,
    ignore_index=True
)

print("Merged dataset shape:", df.shape)
display(df.head())

Merged dataset shape: (8175, 4)


,utterance,intent,category,tags
0,I want help to cancel the order I have made,cancel_order,ORDER,B
1,where can I get information about order cancel...,cancel_order,ORDER,BIM
2,help to cancel the last order,cancel_order,ORDER,B
3,I need help with canceling an order I made,cancel_order,ORDER,B
4,I do not know how I can cancel the order I made,cancel_order,ORDER,BE


In [6]:
print("Column names:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()

print("\nFirst five rows:")
display(df.head())

print("\nMissing values:")
print(df.isnull().sum())

Column names:
['utterance', 'intent', 'category', 'tags']

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8175 entries, 0 to 8174
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   utterance  8175 non-null   object
 1   intent     8175 non-null   object
 2   category   8175 non-null   object
 3   tags       8175 non-null   object
dtypes: object(4)
memory usage: 255.6+ KB

First five rows:


,utterance,intent,category,tags
0,I want help to cancel the order I have made,cancel_order,ORDER,B
1,where can I get information about order cancel...,cancel_order,ORDER,BIM
2,help to cancel the last order,cancel_order,ORDER,B
3,I need help with canceling an order I made,cancel_order,ORDER,B
4,I do not know how I can cancel the order I made,cancel_order,ORDER,BE



Missing values:
utterance    0
intent       0
category     0
tags         0
dtype: int64


In [7]:
df.to_csv(
    "/home/user/Downloads/merged_customer_service_dataset.csv",
    index=False
)

print("Merged dataset saved successfully!")

Merged dataset saved successfully!


In [8]:
print(df.head())
print(df["intent"].value_counts().head(10))

                                           utterance        intent category  \
0        I want help to cancel the order I have made  cancel_order    ORDER   
1  where can I get information about order cancel...  cancel_order    ORDER   
2                      help to cancel the last order  cancel_order    ORDER   
3         I need help with canceling an order I made  cancel_order    ORDER   
4    I do not know how I can cancel the order I made  cancel_order    ORDER   

  tags  
0    B  
1  BIM  
2    B  
3    B  
4   BE  
intent
get_invoice                324
check_invoice              324
payment_issue              323
review                     315
track_refund               308
set_up_shipping_address    307
place_order                306
track_order                305
cancel_order               305
change_order               304
Name: count, dtype: int64


In [9]:
print("Dataset shape:", df.shape)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nUnique intents:", df["intent"].nunique())

Dataset shape: (8175, 4)

Missing values:
utterance    0
intent       0
category     0
tags         0
dtype: int64

Duplicate rows: 0

Unique intents: 27


In [10]:
# Keep the columns required for classification
df = df[["utterance", "intent"]].copy()

# Remove rows with missing text or labels
df = df.dropna(subset=["utterance", "intent"])

# Remove leading and trailing spaces
df["utterance"] = df["utterance"].str.strip()
df["intent"] = df["intent"].str.strip()

# Remove empty text and empty labels
df = df[
    (df["utterance"] != "") &
    (df["intent"] != "")
].copy()

# Remove exact duplicate text-label pairs
df = df.drop_duplicates(subset=["utterance", "intent"])

print("Cleaned dataset shape:", df.shape)

Cleaned dataset shape: (8175, 2)


In [11]:
X = df["utterance"]
y = df["intent"]

print("Input examples:")
print(X.head())

print("\nTarget examples:")
print(y.head())

Input examples:
0          I want help to cancel the order I have made
1    where can I get information about order cancel...
2                        help to cancel the last order
3           I need help with canceling an order I made
4      I do not know how I can cancel the order I made
Name: utterance, dtype: object

Target examples:
0    cancel_order
1    cancel_order
2    cancel_order
3    cancel_order
4    cancel_order
Name: intent, dtype: object


In [12]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))

Training records: 6540
Testing records: 1635


In [13]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2)
    )),
    ("classifier", LogisticRegression(
        max_iter=1000
    ))
])

model.fit(X_train, y_train)

print("Model training completed!")

Model training completed!


In [14]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

Accuracy: 0.9932721712538226

Classification Report:
                          precision    recall  f1-score   support

            cancel_order       0.95      0.98      0.97        61
            change_order       1.00      0.98      0.99        61
 change_shipping_address       0.97      1.00      0.98        59
  check_cancellation_fee       0.98      1.00      0.99        60
           check_invoice       1.00      1.00      1.00        65
   check_payment_methods       1.00      0.97      0.98        59
     check_refund_policy       0.98      1.00      0.99        60
               complaint       1.00      1.00      1.00        60
contact_customer_service       1.00      1.00      1.00        60
     contact_human_agent       1.00      1.00      1.00        59
          create_account       0.97      0.98      0.98        60
          delete_account       1.00      1.00      1.00        60
        delivery_options       1.00      1.00      1.00        60
         delivery_peri

In [15]:
def predict_intent(text):
    prediction = model.predict([text])[0]
    print("Customer message:", text)
    print("Predicted intent:", prediction)

predict_intent("i want to reorder the item  ")

Customer message: i want to reorder the item  
Predicted intent: delivery_period


In [18]:
import joblib

joblib.dump(model, "intent_model.pkl")

print("Model saved successfully!")

Model saved successfully!


In [19]:
print("model" in globals())

True


In [20]:
print(model)

Pipeline(steps=[('tfidf', TfidfVectorizer(ngram_range=(1, 2))),
                ('classifier', LogisticRegression(max_iter=1000))])


In [21]:
import joblib

joblib.dump(model, "intent_model.pkl")

print("Model saved successfully!")

Model saved successfully!


In [22]:
import os

print("Current folder:", os.getcwd())
print("Model file exists:", os.path.exists("intent_model.pkl"))

Current folder: /home/user/Downloads
Model file exists: True
